# Workshop: Governance & Security — Guide

## Scenario

> *"Before go-live: secure RetailHub data with Column Masks, Row Filters, GRANT/REVOKE/DENY, and INFORMATION_SCHEMA."*

You work as a data engineer at **RetailHub**. The platform team is finalising Unity Catalog governance before the production launch. Your job is to implement fine-grained access controls on the Silver layer: grant privileges to the analyst group, mask PII emails, restrict order visibility by region, audit permissions via INFORMATION_SCHEMA, and clean up when done.

## Key Concepts

### Unity Catalog Privilege Model — Three Levels
Access requires three stacked grants: `USE CATALOG` → `USE SCHEMA` → privilege on table/schema.
```sql
GRANT USE CATALOG ON CATALOG my_catalog TO `analysts`;
GRANT USE SCHEMA  ON SCHEMA  my_catalog.silver TO `analysts`;
GRANT SELECT      ON SCHEMA  my_catalog.silver TO `analysts`;
```

### GRANT / REVOKE / DENY
| Statement | Effect |
|-----------|--------|
| `GRANT privilege ON object TO principal` | Allows access |
| `REVOKE privilege ON object FROM principal` | Removes a previously granted privilege |
| `DENY privilege ON object TO principal` | Explicitly blocks access — **overrides any GRANT** |

### Column Mask Function
A SQL UDF assigned to a column. The function receives the raw value and returns either the real value or a masked version based on the caller's identity.
```sql
CREATE OR REPLACE FUNCTION catalog.schema.mask_email(email STRING)
RETURNS STRING
RETURN CASE
    WHEN is_account_group_member('admins') THEN email
    ELSE CONCAT(LEFT(email, 2), '***@***.***')
END;

-- Apply / remove:
ALTER TABLE ... ALTER COLUMN email SET MASK catalog.schema.mask_email;
ALTER TABLE ... ALTER COLUMN email DROP MASK;
```

### Row Filter Function
A SQL UDF that returns `BOOLEAN`. Rows where the function returns `FALSE` are hidden from the caller.
```sql
CREATE OR REPLACE FUNCTION catalog.schema.region_filter(region STRING)
RETURNS BOOLEAN
RETURN (
    is_account_group_member('admins')
    OR (is_account_group_member('east_team') AND region = 'East')
    OR (is_account_group_member('west_team') AND region = 'West')
);

-- Apply / remove:
ALTER TABLE ... SET ROW FILTER catalog.schema.region_filter ON (store_region);
ALTER TABLE ... DROP ROW FILTER;
```

### INFORMATION_SCHEMA Views
Two key views for auditing (scoped to the catalog):
| View | Purpose | Key columns |
|------|---------|-------------|
| `INFORMATION_SCHEMA.TABLES` | List tables/views | `table_schema`, `table_name`, `table_type` |
| `INFORMATION_SCHEMA.TABLE_PRIVILEGES` | List granted privileges | `grantor`, `grantee`, `table_name`, `privilege_type` |

## Prerequisites

- Cluster running and attached to notebook
- `%run ../../setup/00_setup` executed — sets `CATALOG`, `SILVER_SCHEMA`, `BRONZE_SCHEMA` variables
- Setup cells recreate Bronze and Silver `customers` and `orders` tables if they do not exist

## Tasks Overview

Open **`lab_10_governance.ipynb`** and complete the `# TODO` cells.

| Task | What to do | Key concept |
|------|-----------|-------------|
| **Task 1** | Grant privileges to `analysts` group | `GRANT USE CATALOG / USE SCHEMA / SELECT` |
| **Task 2** | List Silver tables via metadata | `INFORMATION_SCHEMA.TABLES` |
| **Task 3** | Create email masking function | Column mask UDF + `is_account_group_member()` |
| **Task 4** | Apply column mask to `customers.email` | `ALTER TABLE ... ALTER COLUMN email SET MASK` |
| **Task 5** | Create region-based row filter function | Row filter UDF returning `BOOLEAN` |
| **Task 6** | Apply row filter to `orders` | `ALTER TABLE ... SET ROW FILTER ... ON (column)` |
| **Task 7** | Audit granted privileges | `INFORMATION_SCHEMA.TABLE_PRIVILEGES` |
| **Task 8** | Remove mask and filter (cleanup) | `DROP MASK`, `DROP ROW FILTER` |
| **Task 9** | Block `contractors` with DENY | `DENY SELECT ON TABLE ... TO principal` |

## Hints

### Task 1: Grant Privileges
All three grants are required. Missing any one prevents the `analysts` group from querying tables.
```python
spark.sql(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `analysts`")
spark.sql(f"GRANT USE SCHEMA  ON SCHEMA  {CATALOG}.{SILVER_SCHEMA} TO `analysts`")
spark.sql(f"GRANT SELECT      ON SCHEMA  {CATALOG}.{SILVER_SCHEMA} TO `analysts`")
```

---

### Task 2: INFORMATION_SCHEMA.TABLES
Query the `TABLES` view scoped to your catalog. Use `table_schema` to filter by schema name.
```python
tables_df = spark.sql(f"""
    SELECT table_schema, table_name, table_type
    FROM {CATALOG}.INFORMATION_SCHEMA.TABLES
    WHERE table_schema = '{SILVER_SCHEMA}'
""")
```

---

### Task 3: Create Column Mask Function
The function takes the raw column value as its only argument. Use `is_account_group_member()` to branch by group.
```python
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{SILVER_SCHEMA}.mask_email(email STRING)
    RETURNS STRING
    RETURN CASE
        WHEN is_account_group_member('admins') THEN email
        ELSE CONCAT(LEFT(email, 2), '***@***.***')
    END
""")
```

---

### Task 4: Apply Column Mask
Reference the function by its full three-part name.
```python
spark.sql(f"""
    ALTER TABLE {CATALOG}.{SILVER_SCHEMA}.customers
    ALTER COLUMN email SET MASK {CATALOG}.{SILVER_SCHEMA}.mask_email
""")
```

---

### Task 5: Create Row Filter Function
The parameter name must match the column name used in the `ON (column)` clause (Task 6).  
The lab uses `store_region` values: `'East'`, `'West'`, `'Central'`.
```python
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{SILVER_SCHEMA}.region_filter(region STRING)
    RETURNS BOOLEAN
    RETURN (
        is_account_group_member('admins')
        OR (is_account_group_member('us_team') AND region = 'East')
        OR (is_account_group_member('eu_team') AND region = 'West')
    )
""")
```

---

### Task 6: Apply Row Filter
The column in `ON (...)` must exist in the table. It is passed as the argument to the filter function.
```python
spark.sql(f"""
    ALTER TABLE {CATALOG}.{SILVER_SCHEMA}.orders
    SET ROW FILTER {CATALOG}.{SILVER_SCHEMA}.region_filter ON (store_region)
""")
```

---

### Task 7: TABLE_PRIVILEGES
The view lists all privileges granted within the catalog.
```python
privs_df = spark.sql(f"""
    SELECT grantor, grantee, table_schema, table_name, privilege_type
    FROM {CATALOG}.INFORMATION_SCHEMA.TABLE_PRIVILEGES
    ORDER BY grantee, table_name
""")
```

---

### Task 8: Remove Mask and Filter
Two separate `ALTER TABLE` statements — one for the column mask, one for the row filter.
```python
spark.sql(f"""
    ALTER TABLE {CATALOG}.{SILVER_SCHEMA}.customers
    ALTER COLUMN email DROP MASK
""")

spark.sql(f"""
    ALTER TABLE {CATALOG}.{SILVER_SCHEMA}.orders
    DROP ROW FILTER
""")
```

---

### Task 9: DENY Privilege
`DENY` is applied at object level and overrides any `GRANT` on the same object, even if the principal also has a schema-level `GRANT SELECT`.
```python
spark.sql(f"""
    DENY SELECT ON TABLE {CATALOG}.{BRONZE_SCHEMA}.orders TO `contractors`
""")

# Verify — SHOW GRANTS lists both GRANT and DENY entries:
spark.sql(f"SHOW GRANTS ON TABLE {CATALOG}.{BRONZE_SCHEMA}.orders").display()
```
> **Note:** The code in Task 9 is commented out because `contractors` must exist as a group. Uncomment to test in an environment where the group exists.

## Summary

| Task | Concept | Key SQL |
|------|---------|---------|
| 1 | Privileges | `GRANT USE CATALOG / USE SCHEMA / SELECT` |
| 2 | Metadata discovery | `INFORMATION_SCHEMA.TABLES` |
| 3–4 | Column Mask | `CREATE FUNCTION … RETURN CASE … SET MASK fn` |
| 5–6 | Row Filter | `CREATE FUNCTION … RETURNS BOOLEAN … SET ROW FILTER fn ON (col)` |
| 7 | Privilege audit | `INFORMATION_SCHEMA.TABLE_PRIVILEGES` |
| 8 | Cleanup | `DROP MASK`, `DROP ROW FILTER` |
| 9 | DENY | `DENY SELECT ON TABLE … TO principal` |

> **Exam Tip:** Three-level access hierarchy is required: `USE CATALOG` + `USE SCHEMA` + data privilege (`SELECT`, `MODIFY`, etc.). Column masks and row filters are SQL UDFs — column mask receives the column value; row filter returns `BOOLEAN`. `DENY` always takes precedence over `GRANT` on the same object. `INFORMATION_SCHEMA` is the standard way to discover and audit metadata within a catalog.

**Congratulations — you have completed all 9 governance tasks and the full Databricks Data Engineer Associate lab series.**